### Shouldnt this be run using the experiment metrics we decided?

## Step 1 — Setup and parameters

We simulate 2000 users (1000 treated, 1000 control). `mu` is average daily watch time in seconds (~95 min). `sigma` is how spread out users are. `rho` is the key knob: how correlated a user's behavior is week-to-week (0.7 = quite predictable). `true_effect` is the real lift the feature causes (+285 sec ≈ +5%) — we *know* it because we're simulating, so we can check which method recovers it best.

In [1]:
import numpy as np 
import pandas as pd 
from scipy import stats 
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

np.random.seed(42) # reproducible

n = 1000 #number of users per arm
mu = 5700 #average daily watch time (95 min) in seconds
sigma = 1800 #std dev across users
rho = 0.7 #pre/post correlation
true_effect = 285 #~5% lift

/Users/shubhisaxena/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/shubhisaxena/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


## Step 2 — Simulate the data

Plain English:
- `Y_pre`  = each user's watch time *last* week (before the experiment).
- `T`      = 1 if treated (sees the leaderboard), 0 if control.
- `Y_post` = watch time *this* week. It's built from three pieces:
  - `rho * (Y_pre - mu)` → the **predictable** part (heavy watchers stay heavy),
  - `true_effect * T`    → the **treatment** bump (only for treated users),
  - `epsilon`            → the **surprise** / random noise.

Why `epsilon`'s size is `sigma*sqrt(1-rho**2)`: it makes `Y_post` end up on the same scale as `Y_pre` **and** makes the realized correlation come out to exactly `rho`. (That's the only "trick" line — you don't need to derive it to use it.)

In [2]:
Y_pre = np.random.normal(mu, sigma, n * 2) #generates random numbers from normal distribution with mean Mu and std dev sigma
T       = np.array([1] * n + [0] * n)
epsilon = np.random.normal(0, sigma * np.sqrt(1 - rho**2), n * 2)

Y_post  = mu + (rho * (Y_pre - mu)) + (true_effect * T)+ epsilon #epsilon is random noise; true_effect * T for 1*true_effect for treatment, rho * (Y_pre - mu) is Theta(Y_pre - mu)
df = pd.DataFrame({'user_id': range(n * 2), 'T': T, 'Y_pre': Y_pre, 'Y_post': Y_post})
df.head()

,user_id,T,Y_pre,Y_post
0,0,1,6594.085475,5742.947114
1,1,1,5451.124258,5625.014427
2,2,1,6865.839369,5782.465731
3,3,1,8441.453742,7508.146279
4,4,1,5278.523925,3255.806297


## Step 3 — Sanity checks

Before trusting any result, confirm the data looks like we intended:
- correlation between pre and post should be ≈ 0.7
- the gap in mean `Y_post` between treated and control should be ≈ 285 (the true effect)

**Gotcha:** always write `df['T']`, never `df.T`. In pandas `.T` means *transpose* the whole table, which silently gives wrong (NaN) answers.

In [3]:
corr = np.corrcoef(df['Y_pre'], df['Y_post'])[0, 1]
mean_t = df[df['T'] == 1]['Y_post'].mean()
mean_c = df[df['T'] == 0]['Y_post'].mean()

print(f"corr(Y_pre, Y_post) : {corr:.3f}   (target ~0.90)")
print(f"mean Y_post treated : {mean_t:.1f}")
print(f"mean Y_post control : {mean_c:.1f}")
print(f"naive gap           : {mean_t - mean_c:.1f}   (true effect = {true_effect})")

corr(Y_pre, Y_post) : 0.683   (target ~0.90)
mean Y_post treated : 6016.9
mean Y_post control : 5765.2
naive gap           : 251.7   (true effect = 285)


## Step 4 — Estimator 1: Standard t-test (the baseline)

The ordinary A/B test. We just compare average `Y_post` in treated vs control.

- **effect** = mean(treated) − mean(control)
- **SE** (standard error) = how much that gap would bounce if we reran the experiment. Formula: `std(Y_post) * sqrt(2/n)`.
- **95% CI** = effect ± 1.96 × SE. If this interval doesn't contain 0, the result is "significant."

This is our reference point. It's *unbiased* (correct on average) but *noisy*. Watch the width of the CI — CUPED will shrink it.

In [8]:
# Estimator 1: Standard t-test
effect_standard = df[df['T']==1]['Y_post'].mean() - df[df['T']==0]['Y_post'].mean()
se_standard     = df['Y_post'].std() * np.sqrt(2/n)
t_stat, p_val   = stats.ttest_ind(df[df['T']==1]['Y_post'], df[df['T']==0]['Y_post']) #Performs standard 2 sample t-test. It evaluates the null hypothesis that two independent groups have identical population means.
ci_standard     = (effect_standard - 1.96*se_standard, effect_standard + 1.96*se_standard)

print("STANDARD T-TEST")
print(f"  effect estimate : {effect_standard:7.1f}   (true = {true_effect})")
print(f"  standard error  : {se_standard:7.1f}")
print(f"  95% CI          : ({ci_standard[0]:.1f}, {ci_standard[1]:.1f})")
print(f"  p-value         : {p_val:.4f}")

STANDARD T-TEST
  effect estimate :   251.7   (true = 285)
  standard error  :    79.7
  95% CI          : (95.4, 408.0)
  p-value         : 0.0016


that means the true effect is between 95.4 to 408 which is a huge spread and we dont know how much lift that is. Too much variance in this normal AB test

## Step 5 — Estimator 2: CUPED

The idea in one sentence: **subtract off the part of each user's `Y_post` that we could have predicted from their `Y_pre`, then run the same t-test on what's left.**

Three lines:
1. **theta** = `Cov(Y_post, Y_pre) / Var(Y_pre)` — this is literally the slope from regressing post on pre. With ρ≈0.7 it should come out near **0.7**.
2. **Y_cuped** = `Y_post − theta·(Y_pre − mean(Y_pre))` — the adjusted outcome. Same average as Y_post (we didn't move the estimate), but less noise.
3. Run the ordinary t-test on `Y_cuped` instead of `Y_post`.

The effect estimate stays ≈ the same (still unbiased). The SE and CI **shrink**, because we deleted the predictable variance.

In [7]:
# Estimator 2: CUPED
theta = np.cov(df['Y_post'], df['Y_pre'])[0,1] / np.var(df['Y_pre'])
df['Y_cuped'] = df['Y_post'] - theta * (df['Y_pre'] - df['Y_pre'].mean())

effect_cuped = df[df['T']==1]['Y_cuped'].mean() - df[df['T']==0]['Y_cuped'].mean()
se_cuped     = df['Y_cuped'].std() * np.sqrt(2/n)
ci_cuped     = (effect_cuped - 1.96*se_cuped, effect_cuped + 1.96*se_cuped)

print(f"theta (should be ~rho=0.7) : {theta:.3f}\n")
print("CUPED")
print(f"  effect estimate : {effect_cuped:7.1f}   (true = {true_effect})")
print(f"  standard error  : {se_cuped:7.1f}   (standard was {se_standard:.1f})")
print(f"  95% CI          : ({ci_cuped[0]:.1f}, {ci_cuped[1]:.1f})")
print(f"  SE reduction    : {100*(1 - se_cuped/se_standard):.1f}%   (theory: {100*(1-np.sqrt(1-rho**2)):.1f}%)")

theta (should be ~rho=0.7) : 0.685

CUPED
  effect estimate :   315.2   (true = 285)
  standard error  :    58.2   (standard was 79.7)
  95% CI          : (201.1, 429.3)
  SE reduction    : 27.0%   (theory: 28.6%)


So we reduced the variance through CUPED and see a smaller CI band along with SE reduction

## Step 6 — Estimator 3: The Lin estimator

CUPED assumes the covariate has the *same* relationship to the outcome in both arms. The **Lin estimator** (Winston Lin, 2013) relaxes that: regress `Y_post` on treatment, the **demeaned** covariate, AND their **interaction**. This lets the pre/post slope differ between treated and control.

```
Y_post ~ T + X_demeaned + T:X_demeaned
```

- `X_demeaned` = `Y_pre − mean(Y_pre)`  (demeaning is what makes the `T` coefficient read as the average treatment effect)
- The **treatment effect** is the coefficient on `T`; its **SE** comes straight from the regression.

Because it's strictly more flexible, Lin is guaranteed to be **at least as efficient as CUPED** (SE ≤ CUPED's), and never hurts. In our simple simulation the two slopes are actually equal, so Lin and CUPED will land very close — but Lin is the safer default in real data.

In [9]:
# Estimator 3: Lin estimator
df['X_demeaned'] = df['Y_pre'] - df['Y_pre'].mean()
lin = smf.ols('Y_post ~ T + X_demeaned + T:X_demeaned', data=df).fit()

effect_lin = lin.params['T']
se_lin     = lin.bse['T']
ci_lin     = (effect_lin - 1.96*se_lin, effect_lin + 1.96*se_lin)

print("LIN ESTIMATOR")
print(f"  effect estimate : {effect_lin:7.1f}   (true = {true_effect})")
print(f"  standard error  : {se_lin:7.1f}")
print(f"  95% CI          : ({ci_lin[0]:.1f}, {ci_lin[1]:.1f})")

LIN ESTIMATOR
  effect estimate :   315.4   (true = 285)
  standard error  :    57.8
  95% CI          : (202.1, 428.7)


## Step 7 — Results comparison table

All three side by side. Watch the SE column shrink and the CI narrow as we go from Standard → CUPED → Lin.

In [10]:
results = pd.DataFrame({
    'Estimator':  ['Standard t-test', 'CUPED', 'Lin estimator'],
    'Effect':     [effect_standard, effect_cuped, effect_lin],
    'SE':         [se_standard, se_cuped, se_lin],
    'CI_low':     [ci_standard[0], ci_cuped[0], ci_lin[0]],
    'CI_high':    [ci_standard[1], ci_cuped[1], ci_lin[1]],
})
results['CI_width']    = results['CI_high'] - results['CI_low']
results['SE_vs_base']  = (results['SE'] / se_standard * 100).round(1).astype(str) + '%'
results['Var_reduction'] = (100*(1 - (results['SE']/se_standard)**2)).round(1).astype(str) + '%'
results.round(1)

,Estimator,Effect,SE,CI_low,CI_high,CI_width,SE_vs_base,Var_reduction
0,Standard t-test,251.7,79.7,95.4,408.0,312.6,100.0%,0.0%
1,CUPED,315.2,58.2,201.1,429.3,228.2,73.0%,46.7%
2,Lin estimator,315.4,57.8,202.1,428.7,226.6,72.5%,47.5%
